# 세션 4 — 맥락 추가(Contextual)와 재정렬(Rerank)

검색 정확도를 두 가지로 더 끌어올립니다. 네 추출본 각각에 원본 / Contextual / Contextual+Rerank 를
적용해 4 x 3 = 12조합을 비교합니다.

- Contextual Retrieval: 청크 앞에 "무슨 내용인지" 한 문장을 LLM으로 붙여 임베딩
- Reranking: 20개를 넉넉히 찾은 뒤 Cross-Encoder로 다시 줄세워 상위 5개만 사용

In [1]:
# 준비 셀 — 경로와 API 키 (매 세션 같은 코드로 시작한다)
import os, time, pickle
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()                            # .env 의 OPENAI_API_KEY 를 환경변수로 등록

ROOT = Path.cwd().parent                 # notebooks/ 의 상위 = 프로젝트 폴더
PDF = ROOT / "data" / "저출생_반전을_위한_대책_관계부처_합동_.pdf"
CACHE = ROOT / "cache"
CACHE.mkdir(exist_ok=True)

print("OpenAI key:", bool(os.getenv("OPENAI_API_KEY")), "| PDF:", PDF.exists())

OpenAI key: True | PDF: True


In [2]:
# 임베딩 모델과 LLM 준비 (세션 2와 동일)
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI

# bge-m3 임베딩: 문장 → 1024차원 벡터 (처음 한 번만 약 2GB 다운로드)
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},    # 인덱스를 만들 때와 같은 설정이어야 한다
)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [3]:
# 공통 헬퍼 — 검색 → 프롬프트 → 생성 (세션 2에서 자세히 설명)
RAG_PROMPT = """다음 맥락을 바탕으로 질문에 답하세요. 맥락에 없는 내용은 추측하지 마세요.

맥락:
{context}

질문: {question}
답변:"""

def ask(question, retriever):
    # 질문과 비슷한 청크를 찾아, 그 내용을 근거로 LLM이 답한다
    docs = retriever.invoke(question)
    context = "\n---\n".join(d.page_content for d in docs)
    answer = llm.invoke(RAG_PROMPT.format(context=context, question=question)).content
    return answer, docs

def show_docs(docs, title="검색 결과"):
    # 검색된 청크를 쪽 번호와 함께 보기 좋게 출력한다
    print(title)
    for i, d in enumerate(docs, 1):
        page = d.metadata.get("page", "?")
        print(f"  {i}. (p.{page}) {d.page_content[:100].strip()} ...")

In [4]:
# 정답이 문서에서 확인된 평가 문항 12개 (세션 1에서 만든 것)
EVAL = [
    {"q": "2023년 한국 합계출산율은?",                         "a": "0.72명"},
    {"q": "2023년 출생아 수는?",                               "a": "23만명"},
    {"q": "정부가 2030년까지 목표로 하는 합계출산율은?",        "a": "1.0"},
    {"q": "육아휴직 급여 최대 상한은 얼마로 인상되나?",         "a": "250만원"},
    {"q": "아빠(배우자) 출산휴가 기간은 며칠로 확대되나?",      "a": "20일"},
    {"q": "부모가 모두 3개월 이상 육아휴직 시 총 육아휴직 기간은?", "a": "1년 6개월"},
    {"q": "대체인력지원금은 월 얼마로 인상되나?",              "a": "120만원"},
    {"q": "임기 내 공공보육 이용률 목표는?",                   "a": "50%"},
    {"q": "0세반 교사 대 영아 비율은 어떻게 개선되나?",        "a": "1:2"},
    {"q": "'25년 이후 출산가구 신생아 특례대출 소득요건은?",    "a": "2.5억원"},
    {"q": "출산가구 대상 주택공급은 연 몇 호로 확대되나?",      "a": "12만호"},
    {"q": "보고서가 제시한 한국의 육아휴직 소득대체율은?",      "a": "38.6%"},
]

def is_correct(answer, expected):
    # 표기 차이를 흡수한다 ("월 250만원" == "250만원", "20근무일" == "20일") — 세션 1에서 설명
    def norm(s):
        s = s.replace(" ", "").replace(",", "").replace("근무일", "일")
        return s[1:] if s.startswith("월") else s
    return norm(expected) in norm(answer)

In [5]:
# 세션 2의 청크와 원본 인덱스 4개를 불러온다
from langchain_community.vectorstores import FAISS

splits_all = pickle.load(open(CACHE / "splits_all.pkl", "rb"))   # {추출본 이름: 청크 리스트}
parsers = list(splits_all)
base_db = {p: FAISS.load_local(str(CACHE / f"idx_{p}"), embeddings,
                               allow_dangerous_deserialization=True) for p in parsers}   # 맥락 없는 원본 인덱스
print({p: len(splits_all[p]) for p in parsers})

REBUILD = False   # True면 4파서 맥락을 직접 생성(본인 키, 청크 합계가 많아 수백 콜)

{'pymupdf': 112, '4llm': 122, 'docling': 140, 'vlm': 181}


## 4-1. 문제: 청크만 보면 무슨 내용인지 모른다

문서를 자르다 보면 이런 청크가 생깁니다 — "1명 자녀 100%, 200만원 / 2명 자녀 100%, 250만원".
**무슨 제도의 얼마인지, 청크만 봐서는 사람도 검색기도 알 수 없습니다.** 섹션 제목과 떨어진 채 잘렸기 때문입니다.

해결책(Anthropic의 Contextual Retrieval): 문서 전체를 아는 LLM에게 **"이 청크가 무슨 내용인지 한 문장"**을
쓰게 해서 청크 앞에 붙인 뒤, 그 상태로 임베딩합니다.

In [ ]:
# 맥락 문장 생성기 — 문서 전체를 아는 LLM이 "이 청크가 무슨 내용인지" 한 문장으로 요약한다
from langchain_core.documents import Document

CONTEXT_PROMPT = """아래 <문서> 를 참고해서, <청크> 가 문서에서 무엇에 대한 내용인지
한국어 한 문장(50~100자)으로 설명해줘. 검색을 돕기 위한 것이니 핵심만.

<문서>
{document}
</문서>

<청크>
{chunk}
</청크>

맥락 문장:"""

def make_context(doc_text, chunk_text):
    # 문서(길면 앞 12000자만)와 청크를 함께 보여 주고 맥락 한 문장을 받아 온다 (LLM 1콜)
    return llm.invoke(CONTEXT_PROMPT.format(document=doc_text[:12000], chunk=chunk_text)).content.strip()

sample_doc = "\n".join(d.page_content for d in splits_all["vlm"])
# 라벨 없이 금액 표만 남은 '맥락 상실' 청크 — 표 일부가 잘려 나온 조각이다
sample = next(d for d in splits_all["vlm"] if d.page_content.startswith("- 통상임금"))
print("원래 청크:", sample.page_content[:130].replace("\n", " "))
print()
print("붙일 맥락:", make_context(sample_doc, sample.page_content))

## 4-2. 네 추출본 모두에 맥락 붙이기

`REBUILD=True` 면 네 추출본 전부에 맥락을 생성해 `cache/contextual_all.pkl` 과 `cache/idx_ctx_*` 로 저장합니다.
청크 합계가 많아 수백 번 호출하므로 한 번만 굽고 이후에는 캐시를 씁니다.

In [ ]:
# 4개 추출본 전체에 맥락을 붙여 새 인덱스(idx_ctx_*)를 만든다 — 캐시가 있으면 로드만
if REBUILD:
    # 느린 길: 청크마다 LLM 1콜 → 총 555콜. 그래서 한 번만 굽고 저장해 둔다
    contextual_all = {}
    for p in parsers:
        doc_text = "\n".join(d.page_content for d in splits_all[p])
        # 핵심 한 줄: [맥락 문장] + 빈 줄 + [원래 청크] 를 이어 붙여 새 Document 로 만든다
        ctx_list = [Document(page_content=make_context(doc_text, d.page_content) + "\n\n" + d.page_content,
                             metadata=d.metadata) for d in splits_all[p]]
        contextual_all[p] = ctx_list
        FAISS.from_documents(ctx_list, embeddings).save_local(str(CACHE / f"idx_ctx_{p}"))   # 맥락 포함 상태로 임베딩
        print(f"{p}: {len(ctx_list)}개")
    pickle.dump(contextual_all, open(CACHE / "contextual_all.pkl", "wb"))
else:
    # 빠른 길: 미리 구워 둔 맥락 청크를 로드만 한다 (수 초)
    contextual_all = pickle.load(open(CACHE / "contextual_all.pkl", "rb"))

# 맥락이 붙은 인덱스 4개 로드 — 이후 "Contextual" 설정은 전부 이걸 쓴다
ctx_db = {p: FAISS.load_local(str(CACHE / f"idx_ctx_{p}"), embeddings,
                              allow_dangerous_deserialization=True) for p in parsers}
print({p: len(contextual_all[p]) for p in parsers})

## 4-3. 맥락 전/후 비교 — 같은 질문, 다른 결과 (docling 기준)

"2030년 목표 합계출산율" 질문으로 비교합니다. 목표치 `1.0` 이 든 청크는 그래프·통계 조각이라
청크만 봐서는 검색기가 알아보지 못합니다 — 맥락 문장이 붙으면 어떻게 달라지는지 봅니다.
(docling은 세션 5·앱에서 최종 채택할 추출본입니다)

In [8]:
# 같은 질문을 원본 인덱스와 맥락 인덱스에 각각 던져, 검색 결과와 답이 어떻게 갈리는지 본다
question = "정부가 2030년까지 목표로 하는 합계출산율은?"

for name, db in [("원본", base_db["docling"]), ("Contextual", ctx_db["docling"])]:
    docs = db.as_retriever(search_kwargs={"k": 5}).invoke(question)
    print(f"[{name}] 검색된 청크 top3")
    for d in docs[:3]:
        print("  -", d.page_content[:75].replace("\n", " "))
    # 검색된 청크로 답까지 생성해 차이를 확인한다
    context = "\n---\n".join(d.page_content for d in docs)
    answer = llm.invoke(RAG_PROMPT.format(context=context, question=question)).content
    print("  → 답변:", answer[:75])
    print()

[원본] 검색된 청크 top3
  - &lt; 합계출산율 &gt;  <!-- image -->  &lt; 출생아수 &gt;  <!-- image -->  - ◇ 금번 대책은
  - ##  (이스라엘) 공동체 가치 중심 의무교육 + 가족중심 문화 등으로 3명대 유지  - ➊ (합계출산율) '21 년 3.00 명 →
  - ## 현 행  | 유형   | 합계출산율   | 비중   | |--------|--------------|--------| | 1유형 


  → 답변: 정부가 2030년까지 목표로 하는 합계출산율에 대한 구체적인 수치는 맥락에 명시되어 있지 않습니다. 따라서 해당 정보는 제공할 수 없습

[Contextual] 검색된 청크 top3
  - 이 청크는 이스라엘과 일본의 저출생 대응 정책을 소개하며, 이스라엘은 공동체 가치 중심의 의무교육과 가족 중심 문화를 통해 높은 출산율
  - 이 청크는 한국의 출산율과 출생아 수가 급격히 감소하고 있는 현황을 설명하며, 특히 2015년부터 2023년까지의 통계와 인구 구조 변
  - 이 청크는 저출생 문제를 해결하기 위한 정책 대전환의 시작점으로, 일·가정 양립, 양육, 주거 등 3대 핵심 분야에 대한 지원과 사회 


  → 답변: 정부가 2030년까지 목표로 하는 합계출산율은 1.0명입니다.



## 4-4. Reranking(재정렬)이란? — 넓게 찾고, 정밀하게 다시 줄세우기

지금까지의 FAISS 검색은 **빠르지만 대충**입니다. 정답 청크를 1위가 아니라 3~8위쯤에 두는 일이 생깁니다.
그래서 상위 5개만 쓰면 정답이 잘려 나갈 수 있습니다. **Reranking은 이 문제를 2단계로 고칩니다.**

**왜 2단계인가 — 검색기와 리랭커는 방식이 다릅니다.**

| | 임베딩 검색 (FAISS) | 리랭커 (Cross-Encoder) |
|---|---|---|
| 방식 | 질문·청크를 **따로** 벡터로 만들어 거리 비교 | 질문·청크를 **함께 읽어** 관련도 채점 |
| 속도 | 빠름 (수천 개도 OK) | 느림 (소수만) |
| 정밀도 | 대충 | 정밀 |

혼자 쓰면 각각 약점이 있으니, **둘을 이어 붙입니다:**

1. **넓게 (FAISS):** 빠른 검색으로 후보 **20개**를 넉넉히 추린다 → *정답을 놓치지 않게(recall)*
2. **정밀하게 (리랭커):** 20개를 질문과 한 쌍씩 다시 채점해 순위를 매긴다 → *진짜 관련된 것만(precision)*
3. **상위 5개만** 남겨 LLM에 넘긴다.

> 도서관 비유: 사서가 키워드로 책 20권을 빠르게 뽑아 온 뒤(1차), 한 권씩 펼쳐 보고
> 질문에 정말 맞는 5권만 골라 주는 것(2차)과 같습니다.

**예시 — "부모가 모두 3개월 이상 육아휴직 시 총 육아휴직 기간은?" (정답: 1년 6개월)**

| 순위 | FAISS (임베딩) | Rerank (리랭커) |
|---|---|---|
| 1위 | 규제 개선 청크 (엉뚱) | **★ 정답 청크 (점수 1.00)** |
| 3위 | ★ 정답 청크 | 유연근무 청크 (0.83) |
| 5위 | (관련 청크) | 규제 개선 청크 (밀려남) |

FAISS는 정답을 **3위**에 두고 엉뚱한 청크를 1위에 뒀지만, 리랭커가 다시 읽어 **정답을 1위로** 끌어올렸습니다.
아래 `run_config` 함수가 이 흐름을 담고, 이어지는 **4-4b 셀**에서 이 재정렬을 직접 눈으로 확인합니다.
(`+Rerank` 갈래의 세 줄이 위 ①넓게 → ②채점 → ③상위 5개에 정확히 대응합니다.)

In [9]:
# 리랭커 준비 + 세 설정(원본/Contextual/Contextual+Rerank)을 실행하는 함수
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")   # 질문·청크를 쌍으로 읽고 정밀 점수를 내는 모델

def run_config(parser, config, question):
    if config == "원본":
        docs = base_db[parser].as_retriever(search_kwargs={"k": 5}).invoke(question)
    elif config == "Contextual":
        docs = ctx_db[parser].as_retriever(search_kwargs={"k": 5}).invoke(question)
    else:
        # Rerank: ① 넓게 20개 후보 → ② 쌍 점수 계산 → ③ 점수순 정렬 후 상위 5개만
        cand = ctx_db[parser].as_retriever(search_kwargs={"k": 20}).invoke(question)
        scores = reranker.predict([[question, d.page_content] for d in cand])   # (질문, 청크) 쌍마다 점수
        docs = [d for _, d in sorted(zip(scores, cand), key=lambda x: x[0], reverse=True)[:5]]
    context = "\n---\n".join(d.page_content for d in docs)
    return llm.invoke(RAG_PROMPT.format(context=context, question=question)).content

## 4-4b. 리랭크가 순위를 바꾸는 장면 보기

앞의 예시를 실제로 확인합니다. 같은 질문에 대해 **FAISS가 매긴 순서**와 **리랭커가 다시 매긴 순서**를
나란히 찍습니다. 정답(1년 6개월)이 든 청크에 `★` 를 붙였습니다 — 순위가 어떻게 뒤바뀌는지 보세요.
(검색·리랭킹은 결정적이라 이 셀은 실행마다 같은 결과가 나옵니다.)

In [32]:
# 4-4b. 리랭크가 순위를 어떻게 바꾸는지 직접 본다 (docling Contextual, 한 질문)
q = "부모가 모두 3개월 이상 육아휴직 시 총 육아휴직 기간은?"   # 정답: 1년 6개월

cand = ctx_db["docling"].as_retriever(search_kwargs={"k": 20}).invoke(q)   # ① FAISS로 넓게 20개
scores = reranker.predict([[q, d.page_content] for d in cand])            # ② (질문,청크) 쌍마다 점수
order = sorted(range(len(cand)), key=lambda i: scores[i], reverse=True)   # ③ 점수 높은 순

def mark(d):      # 정답('1년 6개월')이 든 청크에 ★ 표시 (원문은 띄어쓰기가 껴 있어 공백 제거 후 검사)
    return "★" if "1년6개월" in d.page_content.replace(" ", "") else " "

def snippet(d):   # 정답이 있으면 '그 주변'을, 없으면 청크 앞부분을 보여준다
    t = d.page_content.replace(" ", "").replace("\n", "")
    i = t.find("1년6개월")
    if i >= 0:
        return "…" + t[max(0, i - 14): i + 14] + "…"                        # 정답 주변만 잘라 보이기
    return d.page_content.split("\n\n", 1)[-1].replace("\n", " ").strip()[:38]  # 맥락 문장 뒤 본문 앞부분

print("[FAISS 순서] top5 — 정답(★)이 3위에 묻혀 있다")
for i, d in enumerate(cand[:5], 1):
    print(f"  {i}. {mark(d)} {snippet(d)}")

print("\n[Rerank 순서] top5 — 정답(★)이 1위로, 엉뚱한 FAISS 1위는 아래로")
for rank, i in enumerate(order[:5], 1):
    print(f"  {rank}. {mark(cand[i])} [{scores[i]:.2f}] {snippet(cand[i])}")

[FAISS 순서] top5 — 정답(★)이 3위에 묻혀 있다
  1.   ## 현행  - 통상임금 80%(月150만원 상한) - → 최대 1년
  2.   |                | 현행                 
  3. ★ …직사용시육아휴직기간을1년→1년6개월로연장하여아빠육아…
  4.   ##  수요자 편의를 위한 육아휴직 제도 개선  - ➊ (단기 육아
  5. ★ …시총육아휴직기간연장(1년→1년6개월)-√남성의출산휴…

[Rerank 순서] top5 — 정답(★)이 1위로, 엉뚱한 FAISS 1위는 아래로
  1. ★ [1.00] …직사용시육아휴직기간을1년→1년6개월로연장하여아빠육아…
  2. ★ [0.98] …시총육아휴직기간연장(1년→1년6개월)-√남성의출산휴…
  3.   [0.83] |                | 현행                 
  4.   [0.67] ##  수요자 편의를 위한 육아휴직 제도 개선  - ➊ (단기 육아
  5.   [0.63] ## 현행  - 통상임금 80%(月150만원 상한) - → 최대 1년


## 4-5. 성능 그리드 (12조합 x 12문항)

결과는 `results` 에 저장해 아래 상세에서 다시 씁니다.

> **gpt-4o-mini 144콜 + CPU 리랭킹 48회 — 하루 중 가장 무거운 셀입니다(몇 분, API 과금).**
> 강의에서는 강사가 셀을 걸어 두고 설명을 진행하니, 수강생은 직접 돌리지 말고
> 저장된 출력을 보면 됩니다. 저사양 노트북에서는 리랭킹 때문에 더 오래 걸릴 수 있습니다.

In [10]:
# 12개 설정 조합 x 12문항 = 144콜 + 리랭킹 — 하루 중 가장 무거운 셀 (강사용)
configs = ["원본", "Contextual", "Contextual+Rerank"]
results = {}                              # (추출본, 설정) → 문항별 (문항, 답변, 정오) 목록
for p in parsers:
    for c in configs:
        rows = []
        for it in EVAL:
            answer = run_config(p, c, it["q"])
            rows.append((it, answer, is_correct(answer, it["a"])))
        results[(p, c)] = rows

# 표 출력 — 행: 추출본, 열: 설정, 칸: 점수
print(f"{'파서':<9}", "".join(f"{c:>18}" for c in configs))
for p in parsers:
    row = "".join(f"{str(sum(ok for _,_,ok in results[(p,c)]))+'/12':>18}" for c in configs)
    print(f"{p:<9}", row)

파서                        원본        Contextual Contextual+Rerank
pymupdf                12/12             11/12             12/12
4llm                   11/12             11/12             11/12
docling                11/12             12/12             12/12
vlm                     9/12              9/12              9/12


## 4-6. 한 파서 골라 문항별로 보기

In [11]:
# 추출본 하나를 골라 세 설정의 문항별 답을 비교한다 (저장된 results 재사용 — 추가 호출 없음)
PARSER = "vlm"

print(f"[{PARSER}] 원본 / Contextual / +Rerank")
for i, it in enumerate(EVAL):
    print(f"Q. {it['q']}  (정답: {it['a']})")
    for c in configs:
        ans, ok = results[(PARSER, c)][i][1], results[(PARSER, c)][i][2]
        print(f"   {c:>18} {'O' if ok else 'X'} {ans[:70]}")

[vlm] 원본 / Contextual / +Rerank
Q. 2023년 한국 합계출산율은?  (정답: 0.72명)
                   원본 O 2023년 한국의 합계출산율은 0.72명입니다.
           Contextual O 2023년 한국의 합계출산율은 0.72명입니다.
    Contextual+Rerank O 2023년 한국의 합계출산율은 0.72명입니다.
Q. 2023년 출생아 수는?  (정답: 23만명)
                   원본 O 2023년 출생아 수는 23만명입니다.
           Contextual O 2023년 출생아 수는 23만명입니다.
    Contextual+Rerank O 2023년 출생아 수는 23만명입니다.
Q. 정부가 2030년까지 목표로 하는 합계출산율은?  (정답: 1.0)
                   원본 O 정부가 2030년까지 목표로 하는 합계출산율은 1.0입니다.
           Contextual O 정부가 2030년까지 목표로 하는 합계출산율은 1.0입니다.
    Contextual+Rerank O 정부가 2030년까지 목표로 하는 합계출산율은 1.0입니다.
Q. 육아휴직 급여 최대 상한은 얼마로 인상되나?  (정답: 250만원)
                   원본 O 육아휴직 급여 최대 상한은 150만원에서 250만원으로 인상됩니다.
           Contextual O 육아휴직 급여 최대 상한은 250만원으로 인상됩니다.
    Contextual+Rerank O 육아휴직 급여 최대 상한은 150만원에서 250만원으로 인상됩니다.
Q. 아빠(배우자) 출산휴가 기간은 며칠로 확대되나?  (정답: 20일)
                   원본 O 아빠(배우자) 출산휴가 기간은 10일에서 20일로 확대됩니다.
           Contextual O 아빠(배우자) 출산휴가 기간은 10일에서 20일로 확대됩니다.
    Contextual+Rerank 

## 4-7. 그리드 분석

| 파서 | 원본 | Contextual | Contextual+Rerank |
|---|---|---|---|
| pymupdf | 12 | 11 | 12 |
| 4llm | 11 | 11 | 11 |
| docling | 11 | 12 | 12 |
| vlm | 9 | 9 | 9 |

맥락의 효과는 추출본마다 다릅니다. docling은 11에서 12로 올라가지만, pymupdf는 12에서 11로 살짝 내려갑니다.
이미 깨끗한 추출본에는 맥락이 오히려 약한 노이즈로 작용합니다. 맥락은 청크만으로는 모호한 추출본에서 특히 듣습니다.

Rerank는 맥락이 떨어뜨린 것을 회복시킵니다. pymupdf는 Contextual 11에서 +Rerank 12로 돌아옵니다.
20개를 넓게 본 뒤 정밀하게 재정렬하니 노이즈를 보정하는 안전장치 역할을 합니다.

vlm은 무엇을 해도 9입니다. `2.5억`을 `2~25억`으로 읽는 추출 오류가 청크에 박혀 있어 맥락이든 재정렬이든
그 틀린 청크를 가져오면 똑같이 틀립니다. 검색·후처리는 추출을 넘지 못합니다.

최고 조합이 여러 개(12/12)입니다. 이 강의의 앱(세션 5)은 docling x Contextual+Rerank 를 씁니다.
무료·로컬 추출에 두 기법을 더해 일반 질문에 가장 견고하기 때문입니다. 실제로 앱에서 vlm이 틀리던 `2.5억` 을 정확히 답합니다.

## 직접 해보기

> 예시 코드를 **새 셀에 복사해** 실행해 보세요.

**1. 4-6의 `PARSER` 를 바꿔 맥락·재정렬이 어느 추출본에서 가장 도움이 되는지 보세요.**
```python
PARSER = "docling"                    # 맥락으로 11 → 12가 된 추출본
for i, it in enumerate(EVAL[:3]):     # 앞 3문항만 훑어보기
    print(f"Q. {it['q']}")
    for c in configs:
        ans, ok = results[(PARSER, c)][i][1], results[(PARSER, c)][i][2]
        print(f"   {c:>18} {'O' if ok else 'X'} {ans[:50]}")
```

**2. 한 질문만 골라 세 설정의 답을 직접 비교해 보세요. (`k=20`·`CONTEXT_PROMPT` 를 바꾼 뒤에도)**
```python
q = "대체인력지원금은 월 얼마로 인상되나?"
for c in configs:
    print(f"{c:>18} :", run_config("docling", c, q)[:70])
```

**3. 4-3의 `question` 을 다른 문항으로 바꿔, 맥락이 검색 결과를 언제 바꾸고 언제 안 바꾸는지 보세요.**
```python
question = "아빠(배우자) 출산휴가 기간은 며칠로 확대되나?"
for name, db in [("원본", base_db["docling"]), ("Contextual", ctx_db["docling"])]:
    docs = db.as_retriever(search_kwargs={"k": 3}).invoke(question)
    print(f"[{name}]")
    for d in docs:
        print("  -", d.page_content[:60].replace("\n", " "))
```

In [1]:
PARSER = "docling"                    # 맥락으로 11 → 12가 된 추출본
for i, it in enumerate(EVAL[:3]):     # 앞 3문항만 훑어보기
    print(f"Q. {it['q']}")
    for c in configs:
        ans, ok = results[(PARSER, c)][i][1], results[(PARSER, c)][i][2]
        print(f"   {c:>18} {'O' if ok else 'X'} {ans[:50]}")

NameError: name 'EVAL' is not defined

## 정리
- Contextual은 모호한 청크를 검색이 알아보게 하고, Rerank는 넓게 찾아 정밀하게 줄세웁니다.
- 효과는 추출본·문서에 따라 다릅니다. 만들 때 비용이 크니 한 번 굽고 캐시를 씁니다.
- 다음 세션: 이 검색(docling x Contextual+Rerank)을 챗봇으로 감쌉니다.